In [1]:
# -------------------------------------------------------------------------
# 0. Dependencies Installation
# -------------------------------------------------------------------------
!pip install -q transformers peft bitsandbytes accelerate scikit-learn seaborn matplotlib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 46.0 MB/s eta 0:00:00


In [2]:
# -------------------------------------------------------------------------
# 1. Model 6 Ultimate SOTA Training (Bulletproof OOM Protected)
#    Full 50K Dataset + Disease-Specific Attention Heads + SDPA FlashAttention
#    + Pre-Validation Safe Checkpointing + Expandable Segments
# -------------------------------------------------------------------------
import os, sys, gc, glob, time, json, math, random, torch
from pathlib import Path
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    get_cosine_schedule_with_warmup
)
from peft import (
    LoraConfig, get_peft_model, prepare_model_for_kbit_training,
    set_peft_model_state_dict, get_peft_model_state_dict
)
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

# CUDA Memory Optimization: Prevents fragmentation across long runs
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Set seed for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training Compute Device: {device}", flush=True)

work_dir = "/kaggle/working"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

# -------------------------------------------------------------------------
# Auto-Locate Data Artifacts (Compatible with mimic-cxr-model4-prep-data)
# -------------------------------------------------------------------------
search_dirs = [Path("/kaggle/working"), Path("/kaggle/input")]
manifest_path = None
features_base = None

for base in search_dirs:
    if not base.exists():
        continue
    for root, _, files in os.walk(base):
        if "mimic_master_split_with_rag.csv" in files and manifest_path is None:
            manifest_path = os.path.join(root, "mimic_master_split_with_rag.csv")
            features_base = root
            break
    if manifest_path:
        break

if not manifest_path:
    raise FileNotFoundError(
        "Missing mimic_master_split_with_rag.csv! Please add mimic-cxr-model4-prep-data via '+ Add Input'."
    )

print(f"Found Master Manifest   : {manifest_path}", flush=True)
print(f"Found Features Directory: {features_base}", flush=True)

df_merged = pd.read_csv(manifest_path)
train_df = df_merged[df_merged["split"] == "train"].reset_index(drop=True)
val_df = df_merged[df_merged["split"] == "val"].reset_index(drop=True)
test_df = df_merged[df_merged["split"] == "test"].reset_index(drop=True)

CHEXPERT_COLS = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Enlarged Cardiomediastinum",
    "Fracture", "Lung Lesion", "Lung Opacity", "No Finding", "Pleural Effusion",
    "Pleural Other", "Pneumonia", "Pneumothorax", "Support Devices"
]

# Full 50,176 Training Cohort (Zero artificial sampling cap)
print(f"Dataset Cohort: Full Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}", flush=True)

# Compute class-specific positive weights for Asymmetric Loss
pos_counts = train_df[CHEXPERT_COLS].sum().values
neg_counts = len(train_df) - pos_counts
pos_weights = np.sqrt(neg_counts / np.clip(pos_counts, a_min=1.0, a_max=None))
pos_weights = np.clip(pos_weights, 1.0, 10.0)
pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float, device=device)
print(f"Calculated ASL Class Weights range: min={pos_weights.min():.2f}, max={pos_weights.max():.2f}", flush=True)

# -------------------------------------------------------------------------
# Qwen-2.5-3B Architecture with SDPA FlashAttention & LoRA (r=32, alpha=64)
# -------------------------------------------------------------------------
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_qwen = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
    device_map={"": 0},
    trust_remote_code=True
)

base_qwen = prepare_model_for_kbit_training(base_qwen)

peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
qwen_lora_model = get_peft_model(base_qwen, peft_config)

# -------------------------------------------------------------------------
# Model 6 Breakthrough Architecture: Disease-Specific Attention Pooling
# -------------------------------------------------------------------------
class AttentionPoolClassifier(nn.Module):
    def __init__(self, visual_dim=768, num_classes=14):
        super().__init__()
        self.num_classes = num_classes
        self.visual_dim = visual_dim
        
        # 14 disease queries: learn which patches belong to which disease
        self.attn_proj = nn.Sequential(
            nn.Linear(visual_dim, 256),
            nn.Tanh(),
            nn.Linear(256, num_classes)
        )
        
        # Dedicated classifier heads for each of the 14 diseases
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(visual_dim, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(256, 1)
            ) for _ in range(num_classes)
        ])
        
    def forward(self, spatial_tokens):
        # spatial_tokens: [B, 128, 768]
        attn_scores = self.attn_proj(spatial_tokens)                         # [B, 128, 14]
        attn_weights = F.softmax(attn_scores, dim=1)                          # [B, 128, 14]
        
        # Disease-specific spatial pooling: attend to relevant patches for each class
        # einsum: b=batch, p=128 patches, d=768 dim, c=14 classes
        pooled = torch.einsum('bpd,bpc->bcd', spatial_tokens, attn_weights)   # [B, 14, 768]
        
        # Each disease evaluated by its own dedicated head
        logits_list = [self.heads[c](pooled[:, c, :]) for c in range(self.num_classes)] # 14 x [B, 1]
        logits = torch.cat(logits_list, dim=-1)                               # [B, 14]
        return logits, attn_weights

class MultiTaskQwenModel(nn.Module):
    def __init__(self, qwen_model, visual_dim=768, qwen_dim=2048, num_patches=128):
        super().__init__()
        self.qwen = qwen_model
        self.num_patches = num_patches
        self.qwen_dim = qwen_dim
        
        # Enhanced 2-Layer Visual Connector: 768 -> 2048 -> 2048 with LayerNorm & GELU
        self.visual_projection = nn.Sequential(
            nn.Linear(visual_dim, qwen_dim),
            nn.LayerNorm(qwen_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(qwen_dim, qwen_dim),
            nn.LayerNorm(qwen_dim)
        ).to(device=device)
        
        # Attention-Weighted Pathology Pooling Classifier
        self.attn_pool_classifier = AttentionPoolClassifier(
            visual_dim=visual_dim, num_classes=14
        ).to(device=device)
        
    def build_4d_prefix_attention_mask(self, batch_size, seq_len, device, dtype, text_pad_mask=None):
        total_len = self.num_patches + seq_len
        mask = torch.full((total_len, total_len), -10000.0, dtype=dtype, device=device)
        mask[:self.num_patches, :self.num_patches] = 0.0
        for i in range(self.num_patches, total_len):
            mask[i, :i+1] = 0.0
            
        mask_4d = mask.unsqueeze(0).unsqueeze(0).expand(batch_size, 1, total_len, total_len).clone()
        if text_pad_mask is not None:
            pad_expanded = text_pad_mask.unsqueeze(1).unsqueeze(2)
            mask_4d[:, :, :, self.num_patches:] = mask_4d[:, :, :, self.num_patches:].masked_fill(pad_expanded == 0, -10000.0)
        return mask_4d
        
    def forward(self, spatial_tokens, input_ids, labels=None, text_pad_mask=None, return_attn=False):
        batch_size = spatial_tokens.size(0)
        seq_len = input_ids.size(1)
        
        cls_logits, attn_weights = self.attn_pool_classifier(spatial_tokens.float())
        visual_embeds = self.visual_projection(spatial_tokens.float())
        text_embeds = self.qwen.get_input_embeddings()(input_ids)
        
        if visual_embeds.dtype != text_embeds.dtype:
            visual_embeds = visual_embeds.to(text_embeds.dtype)
            
        inputs_embeds = torch.cat([visual_embeds, text_embeds], dim=1)
        attention_mask_4d = self.build_4d_prefix_attention_mask(
            batch_size, seq_len, spatial_tokens.device, inputs_embeds.dtype, text_pad_mask
        )
        
        if labels is not None:
            vis_labels = torch.full((batch_size, self.num_patches), -100, dtype=labels.dtype, device=labels.device)
            combined_labels = torch.cat([vis_labels, labels], dim=1)
        else:
            combined_labels = None
            
        outputs = self.qwen(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask_4d,
            labels=combined_labels
        )
        if return_attn:
            return outputs.loss, cls_logits, attn_weights
        return outputs.loss, cls_logits

model = MultiTaskQwenModel(qwen_lora_model).to(device)

# -------------------------------------------------------------------------
# Dataset with RAG Dropout, Scheduled Noise & EOS Target (max_seq_len=160)
# -------------------------------------------------------------------------
def find_artifact_file(target_filename, fallback_folder):
    for s_dir in search_dirs:
        if not s_dir.exists():
            continue
        for root, _, files in os.walk(s_dir):
            if target_filename in files:
                return os.path.join(root, target_filename)
    return os.path.join(fallback_folder, target_filename)

class FeaturesDataset(Dataset):
    def __init__(self, df, split_name, total_split_len, tokenizer, max_seq_len=160, max_rag_tokens=40, is_train=True):
        self.df = df
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.max_rag_tokens = max_rag_tokens
        self.is_train = is_train
        
        spatial_file = find_artifact_file(f"features_spatial_{split_name}.dat", features_base)
        print(f"Loading Memory-Mapped Features for [{split_name}] (float16): {spatial_file}", flush=True)
        self.spatial_mmap = np.memmap(spatial_file, dtype="float16", mode="r", shape=(total_split_len, 128, 768))
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        orig_idx = int(row["orig_split_idx"])
        spatial_token = torch.from_numpy(np.array(self.spatial_mmap[orig_idx]))
        
        # Ground Truth CheXpert vector
        cls_labels = torch.tensor([row[c] for c in CHEXPERT_COLS], dtype=torch.float)
        
        # 1. RAG context with 15% RAG Dropout in training (prevents copying laziness)
        retrieved_raw = row["retrieved_report"]
        if self.is_train and random.random() < 0.15:
            retrieved_clean = "None."
        elif pd.isna(retrieved_raw) or str(retrieved_raw).strip().lower() in ["", "nan", "none"]:
            retrieved_clean = "None."
        else:
            retrieved_clean = str(retrieved_raw).strip()
            
        rag_tokens = self.tokenizer.encode(retrieved_clean, add_special_tokens=False)[:self.max_rag_tokens]
        retrieved_clean = self.tokenizer.decode(rag_tokens, skip_special_tokens=True)
        
        # 2. Classifier Conditioning with 8% Scheduled Label Noise
        active_pathologies = [col for col in CHEXPERT_COLS if row[col] == 1.0]
        
        if self.is_train and random.random() < 0.08:
            if active_pathologies and random.random() < 0.5:
                active_pathologies.pop(random.randint(0, len(active_pathologies) - 1))
            else:
                fake_cands = [c for c in CHEXPERT_COLS if c not in active_pathologies]
                if fake_cands:
                    active_pathologies.append(random.choice(fake_cands))
                    
        if not active_pathologies:
            pathology_str = "No acute cardiopulmonary abnormality"
        else:
            pathology_str = ", ".join(active_pathologies)
            
        gt_report = str(row["report"]).strip() if pd.notna(row["report"]) else "Normal chest radiograph."
        
        # 3. Model 6 Classifier-Conditioned Prompt Format WITH Explicit EOS Token Target
        prefix_str = f"Pathology Assessment: {pathology_str}\nReference: {retrieved_clean}\nImpression:"
        full_str = f"{prefix_str} {gt_report}{self.tokenizer.eos_token}"
        
        prefix_tokens = self.tokenizer.encode(prefix_str, add_special_tokens=False)
        prefix_len = len(prefix_tokens)
        
        enc = self.tokenizer(
            full_str,
            truncation=True,
            max_length=self.max_seq_len,
            padding="max_length",
            return_tensors="pt"
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        labels[:min(prefix_len, self.max_seq_len)] = -100
        
        return spatial_token, input_ids, labels, attention_mask, cls_labels

train_ds = FeaturesDataset(train_df, "train", len(train_df), tokenizer, max_seq_len=160, max_rag_tokens=40, is_train=True)
val_ds = FeaturesDataset(val_df, "val", len(val_df), tokenizer, max_seq_len=160, max_rag_tokens=40, is_train=False)

# Safe batch size of 8 for both train and validation: 100% immune to 5GB vocab logit OOM
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=False)

# -------------------------------------------------------------------------
# Asymmetric Loss (ASL) for Multi-Label Clinical Pathology Classification
# -------------------------------------------------------------------------
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4.0, gamma_pos=1.0, clip=0.05, eps=1e-7, pos_weight=None):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps
        self.pos_weight = pos_weight

    def forward(self, logits, targets):
        logits = logits.float()
        targets = targets.float()
        probs = torch.sigmoid(logits).clamp(min=self.eps, max=1.0 - self.eps)
        
        targets_pos = targets
        targets_neg = 1.0 - targets

        probs_neg = probs
        if self.clip is not None and self.clip > 0:
            probs_neg = (probs_neg - self.clip).clamp(min=0.0, max=1.0 - self.eps)

        loss_pos = targets_pos * ((1.0 - probs) ** self.gamma_pos) * torch.log(probs)
        loss_neg = targets_neg * (probs_neg ** self.gamma_neg) * torch.log(1.0 - probs_neg)

        if self.pos_weight is not None:
            loss_pos = loss_pos * self.pos_weight

        loss = -(loss_pos + loss_neg)
        return loss.mean()

criterion_cls = AsymmetricLoss(gamma_neg=4.0, gamma_pos=1.0, clip=0.05, pos_weight=pos_weights_tensor)

# -------------------------------------------------------------------------
# Optimizer & Learning Rate Scheduling
# -------------------------------------------------------------------------
trainable_params = [
    {"params": model.visual_projection.parameters(), "lr": 3e-4, "weight_decay": 0.01},
    {"params": model.attn_pool_classifier.parameters(), "lr": 2e-4, "weight_decay": 0.01},
    {"params": [p for p in model.qwen.parameters() if p.requires_grad], "lr": 5e-5, "weight_decay": 0.01}
]
optimizer = torch.optim.AdamW(trainable_params)

GLOBAL_TOTAL_EPOCHS = 6
EPOCHS_PER_SESSION = 1  # Exactly 1 Epoch per session: safe, predictable, zero timeout risk
accumulation_steps = 4
steps_per_epoch = len(train_loader) // accumulation_steps
total_training_steps = steps_per_epoch * GLOBAL_TOTAL_EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.05 * total_training_steps),
    num_training_steps=total_training_steps
)

# -------------------------------------------------------------------------
# Checkpoint Auto-Resume Mechanism (Cross-Session Seamless Chaining)
# -------------------------------------------------------------------------
start_epoch = 1
best_val_loss = float("inf")

prev_ckpts = glob.glob("/kaggle/input/**/model6_qwen_latest.pt", recursive=True) + \
             glob.glob("/kaggle/input/**/model6_qwen_best.pt", recursive=True) + \
             glob.glob("/kaggle/working/model6_qwen_latest.pt") + \
             glob.glob("/kaggle/working/model6_qwen_best.pt")

resume_ckpt_path = None
if prev_ckpts:
    highest_epoch = -1
    for p in prev_ckpts:
        try:
            temp_ckpt = torch.load(p, map_location="cpu")
            ep = temp_ckpt.get("epoch", 1)
            is_best = "best" in os.path.basename(p).lower()
            if (ep > highest_epoch) or (ep == highest_epoch and is_best):
                highest_epoch = ep
                resume_ckpt_path = p
        except Exception:
            pass

if resume_ckpt_path and os.path.exists(resume_ckpt_path):
    print(f"Resuming training from checkpoint: {resume_ckpt_path}", flush=True)
    ckpt = torch.load(resume_ckpt_path, map_location=device)
    model.visual_projection.load_state_dict(ckpt["visual_projection"])
    model.attn_pool_classifier.load_state_dict(ckpt["attn_pool_classifier"])
    set_peft_model_state_dict(qwen_lora_model, ckpt["lora_state"])
    if "optimizer" in ckpt:
        try:
            optimizer.load_state_dict(ckpt["optimizer"])
        except Exception as e:
            print(f"Warning: could not restore optimizer state ({e})", flush=True)
    if "scheduler" in ckpt:
        try:
            scheduler.load_state_dict(ckpt["scheduler"])
        except Exception as e:
            print(f"Warning: could not restore scheduler state ({e})", flush=True)
    start_epoch = ckpt.get("epoch", 1) + 1
    best_val_loss = ckpt.get("val_loss", float("inf"))
    print(f"Resumed state loaded! Starting at Epoch {start_epoch} (Prior Best Val Loss: {best_val_loss:.4f})", flush=True)
else:
    print("No prior checkpoint found. Initializing Model 6 training from scratch (Epoch 1).", flush=True)

end_epoch = min(start_epoch + EPOCHS_PER_SESSION - 1, GLOBAL_TOTAL_EPOCHS)
print(f"Session Execution Target: Epoch {start_epoch} to Epoch {end_epoch} (Max Global: {GLOBAL_TOTAL_EPOCHS})", flush=True)

# -------------------------------------------------------------------------
# Training Loop with Dynamic Loss Weighting
# -------------------------------------------------------------------------
scaler = torch.amp.GradScaler("cuda")
training_history = []
prior_history_files = glob.glob("/kaggle/input/**/training_history.json", recursive=True)
if prior_history_files and os.path.exists(prior_history_files[0]):
    try:
        with open(prior_history_files[0]) as f:
            training_history = json.load(f)
            print(f"Loaded {len(training_history)} prior epoch history records.", flush=True)
    except Exception:
        pass

for epoch in range(start_epoch, end_epoch + 1):
    epoch_start_time = time.time()
    model.train()
    running_joint_loss, running_text_loss, running_cls_loss = 0.0, 0.0, 0.0
    optimizer.zero_grad()
    
    # Dynamic Loss Weight: 2.0 down to 0.5 across epochs
    cls_weight = max(0.5, 2.0 - 0.3 * (epoch - 1))
    print(f"\n--- Starting Epoch {epoch}/{GLOBAL_TOTAL_EPOCHS} (Classifier Weight: {cls_weight:.2f}) ---", flush=True)
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{end_epoch} [Train]")
    for step, (spatial_t, input_ids, labels, attn_mask, cls_targets) in enumerate(pbar):
        spatial_t = spatial_t.to(device)
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        attn_mask = attn_mask.to(device)
        cls_targets = cls_targets.to(device)
        
        with torch.amp.autocast("cuda"):
            loss_text, cls_logits = model(spatial_t, input_ids, labels=labels, text_pad_mask=attn_mask)
            loss_cls = criterion_cls(cls_logits, cls_targets)
            joint_loss = (loss_text + cls_weight * loss_cls) / accumulation_steps
            
        scaler.scale(joint_loss).backward()
        
        if (step + 1) % accumulation_steps == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            all_trainable = [p for p in model.parameters() if p.requires_grad]
            torch.nn.utils.clip_grad_norm_(all_trainable, max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
            
        running_joint_loss += joint_loss.item() * accumulation_steps
        running_text_loss += loss_text.item()
        running_cls_loss += loss_cls.item()
        
        if (step + 1) % 50 == 0 or (step + 1) == len(train_loader):
            current_lr = scheduler.get_last_lr()[0]
            pbar.set_postfix({
                "Joint": f"{joint_loss.item() * accumulation_steps:.4f}",
                "Text": f"{loss_text.item():.4f}",
                "Cls": f"{loss_cls.item():.4f}",
                "LR": f"{current_lr:.2e}"
            })
            
    epoch_train_joint = running_joint_loss / len(train_loader)
    epoch_train_text = running_text_loss / len(train_loader)
    epoch_train_cls = running_cls_loss / len(train_loader)
    
    # ---------------------------------------------------------------------
    # Immediate Pre-Validation Checkpoint Save (Protects 8 Hours of Training)
    # ---------------------------------------------------------------------
    print(f"\nEpoch {epoch} Training Loop Complete. Saving immediate checkpoint before validation...", flush=True)
    ckpt_state = {
        "visual_projection": model.visual_projection.state_dict(),
        "attn_pool_classifier": model.attn_pool_classifier.state_dict(),
        "lora_state": {k: v.cpu() for k, v in get_peft_model_state_dict(qwen_lora_model).items()},
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "epoch": epoch,
        "train_joint": epoch_train_joint,
        "val_loss": float("inf"),
        "val_macro_f1": 0.0,
        "cls_weight": cls_weight
    }
    torch.save(ckpt_state, "/kaggle/working/model6_qwen_latest.pt")
    print("Pre-validation checkpoint saved to /kaggle/working/model6_qwen_latest.pt", flush=True)

    # ---------------------------------------------------------------------
    # Validation Phase (batch_size=8 + Clean Memory Cache)
    # ---------------------------------------------------------------------
    optimizer.zero_grad(set_to_none=True)
    torch.cuda.empty_cache()
    gc.collect()

    model.eval()
    val_joint_loss, val_text_loss, val_cls_loss = 0.0, 0.0, 0.0
    all_val_probs, all_val_targets = [], []
    
    with torch.no_grad():
        for spatial_t, input_ids, labels, attn_mask, cls_targets in tqdm(val_loader, desc=f"Epoch {epoch}/{end_epoch} [Val]"):
            spatial_t = spatial_t.to(device)
            input_ids = input_ids.to(device)
            labels = labels.to(device)
            attn_mask = attn_mask.to(device)
            cls_targets = cls_targets.to(device)
            
            with torch.amp.autocast("cuda"):
                loss_text, cls_logits = model(spatial_t, input_ids, labels=labels, text_pad_mask=attn_mask)
                loss_cls = criterion_cls(cls_logits, cls_targets)
                joint_loss = loss_text + cls_weight * loss_cls
                
            val_joint_loss += joint_loss.item()
            val_text_loss += loss_text.item()
            val_cls_loss += loss_cls.item()
            
            probs = torch.sigmoid(cls_logits.float()).cpu().numpy()
            all_val_probs.append(probs)
            all_val_targets.append(cls_targets.cpu().numpy())
            
    epoch_val_joint = val_joint_loss / len(val_loader)
    epoch_val_text = val_text_loss / len(val_loader)
    epoch_val_cls = val_cls_loss / len(val_loader)
    
    val_probs_arr = np.vstack(all_val_probs)
    val_targets_arr = np.vstack(all_val_targets)
    
    val_preds_binary = (val_probs_arr >= 0.5).astype(int)
    val_macro_f1 = f1_score(val_targets_arr, val_preds_binary, average="macro", zero_division=0)
    
    elapsed_epoch = (time.time() - epoch_start_time) / 60.0
    print(f"\n--- Epoch {epoch} Summary ({elapsed_epoch:.1f}m) ---", flush=True)
    print(f"Train Loss : Joint={epoch_train_joint:.4f} | Text={epoch_train_text:.4f} | Cls={epoch_train_cls:.4f}", flush=True)
    print(f"Val Loss   : Joint={epoch_val_joint:.4f} | Text={epoch_val_text:.4f} | Cls={epoch_val_cls:.4f}", flush=True)
    print(f"Val Metric : Macro F1 (at 0.5)={val_macro_f1:.4f}", flush=True)
    print("-" * 55, flush=True)
    
    # ---------------------------------------------------------------------
    # Update Checkpoints with Validated Metrics
    # ---------------------------------------------------------------------
    ckpt_state["val_loss"] = epoch_val_joint
    ckpt_state["val_macro_f1"] = val_macro_f1
    torch.save(ckpt_state, "/kaggle/working/model6_qwen_latest.pt")
    
    if epoch_val_joint < best_val_loss:
        best_val_loss = epoch_val_joint
        torch.save(ckpt_state, "/kaggle/working/model6_qwen_best.pt")
        print(f"[*] New best validation loss achieved ({best_val_loss:.4f})! Saved model6_qwen_best.pt", flush=True)
        
    training_history.append({
        "epoch": epoch,
        "train_joint": epoch_train_joint, "train_text": epoch_train_text, "train_cls": epoch_train_cls,
        "val_joint": epoch_val_joint, "val_text": epoch_val_text, "val_cls": epoch_val_cls,
        "val_macro_f1": val_macro_f1, "cls_weight": cls_weight
    })

with open("/kaggle/working/training_history.json", "w") as f:
    json.dump(training_history, f, indent=2)

# Generate and save Learning Curves
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_range = [h["epoch"] for h in training_history]
ax1.plot(epochs_range, [h["train_joint"] for h in training_history], marker="o", label="Train Joint Loss")
ax1.plot(epochs_range, [h["val_joint"] for h in training_history], marker="s", label="Val Joint Loss")
ax1.set_title("Model 6 Joint Loss Progression", fontweight="bold")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.plot(epochs_range, [h["train_text"] for h in training_history], marker="o", label="Train Text Loss")
ax2.plot(epochs_range, [h["val_text"] for h in training_history], marker="s", label="Val Text Loss")
ax2.set_title("Model 6 Text Completion Loss Progression", fontweight="bold")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig("/kaggle/working/learning_curves.png", dpi=300)
plt.close()
print("Learning curves plot saved to /kaggle/working/learning_curves.png", flush=True)

# -------------------------------------------------------------------------
# Dynamic Optimal Threshold Computation on Final Validation Predictions
# -------------------------------------------------------------------------
print("\nComputing Class-Specific Optimal Decision Thresholds on Validation Cohort...", flush=True)
optimal_thresholds = {}
for i, col in enumerate(CHEXPERT_COLS):
    best_f1, best_thresh = 0.0, 0.5
    for thresh in np.arange(0.1, 0.9, 0.02):
        preds = (val_probs_arr[:, i] >= thresh).astype(int)
        f1 = f1_score(val_targets_arr[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh
    optimal_thresholds[col] = float(np.round(best_thresh, 3))
    print(f"  {col:<26} | Optimal Thresh: {best_thresh:.2f} | Val F1: {best_f1:.4f}", flush=True)

with open("/kaggle/working/optimal_thresholds.json", "w") as f:
    json.dump(optimal_thresholds, f, indent=2)

print("\nOptimal Thresholds successfully saved to /kaggle/working/optimal_thresholds.json", flush=True)
print("Model 6 Training Session Completed Successfully!", flush=True)


Training Compute Device: cuda
Found Master Manifest   : /kaggle/input/datasets/mikuchinu/mimic-cxr-model4-prep-data/mimic_master_split_with_rag.csv
Found Features Directory: /kaggle/input/datasets/mikuchinu/mimic-cxr-model4-prep-data
Dataset Cohort: Full Train=50176 | Val=6190 | Test=6388
Calculated ASL Class Weights range: min=1.19, max=10.00


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

... (7 outputs left out)

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)



Epoch 3 Training Loop Complete. Saving immediate checkpoint before validation...
Pre-validation checkpoint saved to /kaggle/working/model6_qwen_latest.pt


Epoch 3/3 [Val]:   0%|          | 0/774 [00:00<?, ?it/s]


--- Epoch 3 Summary (499.4m) ---
Train Loss : Joint=1.1656 | Text=1.0608 | Cls=0.0749
Val Loss   : Joint=1.5046 | Text=1.3888 | Cls=0.0827
Val Metric : Macro F1 (at 0.5)=0.3448
-------------------------------------------------------
Learning curves plot saved to /kaggle/working/learning_curves.png

Computing Class-Specific Optimal Decision Thresholds on Validation Cohort...
  Atelectasis                | Optimal Thresh: 0.62 | Val F1: 0.5013
  Cardiomegaly               | Optimal Thresh: 0.58 | Val F1: 0.4976
  Consolidation              | Optimal Thresh: 0.62 | Val F1: 0.2493
  Edema                      | Optimal Thresh: 0.60 | Val F1: 0.5056
  Enlarged Cardiomediastinum | Optimal Thresh: 0.52 | Val F1: 0.1402
  Fracture                   | Optimal Thresh: 0.58 | Val F1: 0.1667
  Lung Lesion                | Optimal Thresh: 0.60 | Val F1: 0.2526
  Lung Opacity               | Optimal Thresh: 0.62 | Val F1: 0.4606
  No Finding                 | Optimal Thresh: 0.62 | Val F1: 0.7631
 